# M0–M3 reconstruction quality

Run the [quality evaluator](README.md#measure-reconstruction-quality) first.
This notebook reads saved reports and shows corpus BLEU, chrF++, semantic
similarity, ROUGE-L, and BPE token accuracy versus SNR. The shaded bands show
variation across channel trials for the same selected checkpoints.


In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

REPORT_DIR = None  # Or the path to a specific receiver_quality run directory.
EXPORT_DIR = None  # Or a fresh directory for PNG/PDF figures.

start = Path.cwd().resolve()
repo = next((p for p in (start, *start.parents)
             if (p / "benchmarks/receiver/quality_plots.py").is_file()), None)
if repo is None:
    raise FileNotFoundError("Open Jupyter from inside the pecSC repository.")
sys.path.insert(0, str(repo))

from benchmarks.receiver.quality_plots import LABELS, MODELS, plot_quality

required = ("quality_manifest.json", "quality_summary.csv", "paired_differences.csv")
if REPORT_DIR is None:
    candidates = [p.parent for p in (repo / "results/article_kd/receiver_quality").glob("*/quality_manifest.json")
                  if all((p.parent / name).is_file() for name in required)]
    if not candidates:
        raise FileNotFoundError("Run the quality evaluator first or set REPORT_DIR.")
    report_dir = max(candidates, key=lambda p: ((p / required[0]).stat().st_mtime_ns, str(p)))
else:
    report_dir = (repo / Path(REPORT_DIR).expanduser()).resolve()

manifest = json.loads((report_dir / required[0]).read_text(encoding="utf-8"))
summary = pd.read_csv(report_dir / required[1])
differences = pd.read_csv(report_dir / required[2])
protocol = manifest["protocol"]
expected = {(model, snr) for model in MODELS for snr in protocol["snr_db"]}
observed = set(zip(summary["model"], summary["snr_db"]))
if observed != expected or len(summary) != len(expected):
    raise ValueError("The summary must contain one row per model and SNR in the manifest.")
if not summary["trials"].eq(len(protocol["trial_seeds"])).all():
    raise ValueError("The trial count does not match the manifest.")

print(f"Report: {report_dir}")
for warning in manifest.get("protocol_warnings", []):
    print(f"Exploratory validation: {warning}")


In [ ]:
figure = plot_quality(summary, manifest)
plt.show()
display(summary[["model", "snr_db", *[f"{metric}_mean" for metric in LABELS]]].round(4))


Positive deltas favor the candidate over M1. Each difference is taken within
the same SNR and channel trial before averaging. The table is descriptive
for one training seed.


In [ ]:
delta_columns = [f"delta_{metric}" for metric in LABELS]
display(differences.groupby(["model", "snr_db"])[delta_columns].mean().round(4))

if EXPORT_DIR is not None:
    output = (repo / Path(EXPORT_DIR).expanduser()).resolve()
    output.mkdir(parents=True, exist_ok=False)
    figure.savefig(output / "quality_vs_snr.png", dpi=200)
    figure.savefig(output / "quality_vs_snr.pdf")
    print(f"Figures saved to {output}")
